In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [3]:
df = pd.read_csv("solar_system.csv")
print("before:", df.shape)
df = df.set_index("Attribute").T
print("after set_index + transpose:", df.shape)
df.index.name = "planet"
df.reset_index(inplace=True)
df.columns.name = None
print("final shape:", df.shape)

#before it was (20,11) and after its (10,20)
#before it was rows = attributes, columns = plantets 
#after row = planets and columns = attributes 

#set_index makes attribute the row label and .T transposes and df.index.name changes the columns and reset_index() turn index into a column
#its not 11,20 because attribute becomes index and then reset_index adds "Planet" as a new column
print(df.columns)
unit = [col for col in df.columns if "(" in col]
no_unit = [col for col in df.columns if "(" not in col]

print(len(unit), unit)
print(len(no_unit), no_unit)
print(df)

before: (20, 11)
after set_index + transpose: (10, 20)
final shape: (10, 21)
Index(['planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')
16 ['Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Obliquity to Orbit (deg)', 'Mean Temperature 

In [4]:
print(df.dtypes)
for col in df.columns:
    print(col)
    print(df[col].apply(type).value_counts())
#there is str float and a mix
#strings make it slower as like "100" + "200" equals like "100200" instead of "300"

planet                         object
Mass (10^24kg)                 object
Diameter (km)                  object
Density (kg/m^3)               object
Gravity (m/s^2)                object
Escape Velocity (km/s)         object
Rotation Period (hours)        object
Length of Day (hours)          object
Distance from Sun (10^6 km)    object
Perihelion (10^6 km)           object
Aphelion (10^6 km)             object
Orbital Period (days)          object
Orbital Velocity (km/s)        object
Orbital Inclination (deg)      object
Orbital Eccentricity           object
Obliquity to Orbit (deg)       object
Mean Temperature (C)           object
Surface Pressure (bars)        object
Number of Moons                object
Ring System?                   object
Global Magnetic Field?         object
dtype: object
planet
planet
<class 'str'>    10
Name: count, dtype: int64
Mass (10^24kg)
Mass (10^24kg)
<class 'str'>    10
Name: count, dtype: int64
Diameter (km)
Diameter (km)
<class 'str'>    10
Name

In [5]:
for col in df.columns:
    if col not in ["Planet", "Ring System?", "Global Magnetic Field?"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.dtypes)
#the three data types are float64 int64 and object 
# these are decimal whole numbers and strings/mixed 
# the remaining are strings yes/no for ring system and yes/no for global magnetic field 


planet                         float64
Mass (10^24kg)                 float64
Diameter (km)                    int64
Density (kg/m^3)                 int64
Gravity (m/s^2)                float64
Escape Velocity (km/s)         float64
Rotation Period (hours)        float64
Length of Day (hours)          float64
Distance from Sun (10^6 km)    float64
Perihelion (10^6 km)           float64
Aphelion (10^6 km)             float64
Orbital Period (days)          float64
Orbital Velocity (km/s)        float64
Orbital Inclination (deg)      float64
Orbital Eccentricity           float64
Obliquity to Orbit (deg)       float64
Mean Temperature (C)             int64
Surface Pressure (bars)        float64
Number of Moons                  int64
Ring System?                    object
Global Magnetic Field?          object
dtype: object


In [6]:
def attach_units(df, col, unit, new_name):
    df[col] = df[col] * unit
    df.rename(columns={col: new_name}, inplace=True)
attach_units(df, "Mass (10^24kg)", 1e24 * u.kg, "Mass (kg)")
print(df["Mass (kg)"])

0    3.300000e+23
1    4.870000e+24
2    5.970000e+24
3    7.300000e+22
4    6.420000e+23
5    1.898000e+27
6    5.680000e+26
7    8.680000e+25
8    1.020000e+26
9    1.300000e+22
Name: Mass (kg), dtype: float64


In [9]:
semi_major = (df["Perihelion (10^6 km)"] + df["Aphelion (10^6 km)"]) / 2
if "Semi-Major Axis (10^6 km)" in df.columns:
    df.drop(columns=["Semi-Major Axis (10^6 km)"], inplace=True)
df.insert(
    df.columns.get_loc("Aphelion (10^6 km)") + 1,
    "Semi-Major Axis (10^6 km)",
    semi_major
)
print(df.columns)
print(df["Semi-Major Axis (10^6 km)"])

period = df["Orbital Period (days)"].astype(float).values * u.day
df["Orbital Period (years)"] = period.to(u.year).value
print(df["Orbital Period (years)"])

df["planet"] = df["planet"].astype(str).str.strip()
planet = "Earth"
val = df.loc[df["planet"] == planet, "Orbital Period (years)"].values[0]
print(f"{planet}: {val:.4f}")

Index(['planet', 'Mass (kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)',
       'Semi-Major Axis (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?', 'Orbital Period (years)'],
      dtype='object')
0      57.9000
1     108.2000
2     149.6000
3       0.3845
4     228.0000
5     778.5000
6    1432.0500
7    2867.0500
8    4515.0000
9    5906.3500
Name: Semi-Major Axis (10^6 km), dtype: float64
0      0.240931
1      0.615195
2      0.999863
3      0.074743
4      1.880903
5     11.857632
6     29.423682
7     83.748118
8    163.723477
9    247.939767
Name: Orbital Period (

IndexError: index 0 is out of bounds for axis 0 with size 0

In [52]:
#average distance between sun and earth
print(const.au)
print(const.au.to(u.km))
distance_cols = [col for col in df.columns if "(km)" in col]
for col in distance_cols:
    new_col = col.replace("(km)", "(AU)")
    df[new_col] = (df[col] * u.km).to(u.au)
planet = "Earth"
for col in df.columns:
    if "(AU)" in col:
        print(col, df.loc[df["Planet"] == planet, col].values[0])
df.to_csv("units.csv", index=False)

  Name   = Astronomical Unit
  Value  = 149597870700.0
  Uncertainty  = 0.0
  Unit  = m
  Reference = IAU 2012 Resolution B2
149597870.70000002 km


AttributeError: 'Series' object has no attribute 'to'